In [0]:
# ============================================================
# CUSTOMER SILVER
# Bronze → Clean → Validate → SCD Type 2 → Silver
# Incremental + Watermark + Quarantine
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from delta.tables import DeltaTable

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

# IMPORTANT:
# Set True ONLY for the current one-time SCD2 repair.
# After successful repair, change to False.         

# REBUILD_SCD2 = False
PIPELINE_NAME = "customers_silver"
SOURCE_SYSTEM = "customer_api"
VALID_MARKETS = [
    "India",
    "Canada",
    "UK",
    "Australia",
    "Germany"
]

In [0]:



# ============================================================
# 1. PATHS
# ============================================================
bronze_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "bronze/customers/"
)
silver_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "silver/customers/"
)
quarantine_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "quarantine/customers/"
)
control_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)

In [0]:
# ============================================================
# 2. READ BRONZE
# ============================================================

bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)
print("Bronze customer records:", bronze_df.count())

Bronze customer records: 175003


In [0]:
# ============================================================
# 3. TRANSFORMATIONS
# ============================================================

customers_transformed_df = (
    bronze_df
    # Customer ID
    .withColumn(
        "customer_id",
        F.col("customer_id").cast("long")
    )
    # Clean name
    .withColumn(
        "name",
        F.initcap(
            F.trim(
                F.col("name")
            )
        )
    )
    # Clean city
    .withColumn(
        "city",
        F.initcap(
            F.trim(
                F.col("city")
            )
        )
    )
    # Standardize market
    .withColumn(
        "market",
        F.initcap(
            F.trim(
                F.col("market")
            )
        )
    )
)

In [0]:
# ============================================================
# 4. DATA QUALITY VALIDATION
# ============================================================

invalid_condition = (
    F.col("customer_id").isNull()
    |
    F.col("name").isNull()
    |
    F.col("city").isNull()
    |
    F.col("market").isNull()
    # |
    # ~F.col("market").isin(VALID_MARKETS)
)
invalid_customers_df = (
    customers_transformed_df
    .filter(invalid_condition)
)
valid_customers_df = (
    customers_transformed_df
    .filter(~invalid_condition)
)
invalid_count=invalid_customers_df.count()
valid_count=valid_customers_df.count()
print("Valid customers:",valid_count)
print("Invalid customers:",invalid_count)

Valid customers: 175003
Invalid customers: 0


In [0]:
# ============================================================
# 5. WRITE INVALID RECORDS TO QUARANTINE
# ============================================================
if invalid_count > 0:
    quarantine_df = (
        invalid_customers_df
        .withColumn(
            "quarantine_reason",
            F.when(
                F.col("customer_id").isNull(),
                F.lit("NULL_CUSTOMER_ID")
            )
            .when(
                F.col("name").isNull(),
                F.lit("NULL_NAME")
            )
            .when(
                F.col("city").isNull(),
                F.lit("NULL_CITY")
            )
            .when(
                F.col("market").isNull(),
                F.lit("NULL_MARKET")
            )
            .when(
                ~F.col("market").isin(VALID_MARKETS),
                F.lit("INVALID_MARKET")
            )
            .otherwise(
                F.lit("UNKNOWN_VALIDATION_ERROR")
            )
        )
        .withColumn(
            "quarantine_timestamp",
            F.current_timestamp()
        )
    )
    (
        quarantine_df
        .write
        .format("delta")
        .mode("append")
        .save(quarantine_path)
    )
    print(
        "Invalid records written to quarantine:",
        invalid_count
    )          
else:
    print("No invalid customer records.") 

No invalid customer records.


In [0]:
# ============================================================
# 6. REMOVE EXACT DUPLICATES
# ============================================================

valid_customers_df = (
    valid_customers_df
    .dropDuplicates([
        "customer_id",
        "name",
        "city",
        "market"
    ])
)
print(
    "Valid records after exact deduplication:",
    valid_customers_df.count()
)

Valid records after exact deduplication: 173128


In [0]:
# ============================================================
# 7. READ / CREATE WATERMARK
# ============================================================

control_table = DeltaTable.forPath(
    spark,
    control_path
)
control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)                                                         
watermark_exists = (
    control_df
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .count() > 0
)
print(
    "customers_silver watermark exists:",
    watermark_exists
)

customers_silver watermark exists: True


In [0]:
# ============================================================
# 8. CREATE WATERMARK ROW IF MISSING
# ============================================================

watermark_schema = StructType([
    StructField(
        "pipeline_name",
        StringType(),
        False
    ),
    StructField(
        "last_processed_timestamp",
        TimestampType(),
        True
    ),
    StructField(
        "last_run_id",
        StringType(),
        True
    ),
    StructField(
        "updated_at",
        TimestampType(),
        True
    )
])

In [0]:
if not watermark_exists:
    initial_watermark_df = spark.createDataFrame(
        [
            (
                PIPELINE_NAME,
                None,
                None,
                None
            )
        ],
        schema=watermark_schema
    )
    (
        initial_watermark_df
        .write
        .format("delta")
        .mode("append")
        .save(control_path)
    )
    print(
        "Created customers_silver watermark."
    )

In [0]:

# ============================================================
# 9. READ CURRENT WATERMARK
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)


last_processed_timestamp = (
    control_df
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .select(
        "last_processed_timestamp"
    )
    .collect()[0][0]
)


print(
    "Current customers_silver watermark:",
    last_processed_timestamp
)

Current customers_silver watermark: 2026-10-15 04:18:00.581674


In [0]:

# ============================================================
# CUSTOMER SILVER - SCD TYPE 2
# AUTOMATIC INITIAL LOAD + INCREMENTAL PROCESSING
# ============================================================



# ============================================================
# 1. DETECT INITIAL LOAD OR INCREMENTAL LOAD
# ============================================================

silver_exists = DeltaTable.isDeltaTable(
    spark,
    silver_path
)

if not silver_exists and last_processed_timestamp is None:

    load_type = "INITIAL"

    print("Initial Customer Silver load detected.")

    processed_customers_df = valid_customers_df

elif silver_exists and last_processed_timestamp is not None:

    load_type = "INCREMENTAL"

    print("Incremental Customer Silver load detected.")

    processed_customers_df = (
        valid_customers_df
        .filter(
            F.col("ingestion_timestamp")
            > F.lit(last_processed_timestamp)
        )
    )

else:

    raise Exception(
        "Silver table and watermark are inconsistent. "
        "Investigate before processing."
    )


# ============================================================
# 2. CHECK AVAILABLE RECORDS
# ============================================================

processed_count = processed_customers_df.count()

print("Load type:", load_type)
print("Records to process:", processed_count)

if processed_count == 0:

    print("No new customer records.")
    print("Silver is already up to date.")

    dbutils.notebook.exit(
        "SUCCESS: No new customer records."
    )


# ============================================================
# 3. INITIAL LOAD - BUILD COMPLETE AVAILABLE SCD2 HISTORY
# ============================================================

if load_type == "INITIAL":

    # Remove exact duplicate source records.
    rebuild_source_df = (
        processed_customers_df
        .dropDuplicates([
            "customer_id",
            "name",
            "city",
            "market",
            "ingestion_timestamp",
            "source_file"
        ])
    )

    # Define when each version becomes effective.
    rebuild_source_df = (
        rebuild_source_df
        .withColumn(
            "effective_from",
            F.col("ingestion_timestamp")
        )
    )

    # Order all versions belonging to each customer.
    history_window = (
        Window
        .partitionBy("customer_id")
        .orderBy(
            F.col("effective_from").asc(),
            F.col("source_file").asc()
        )
    )

    # Find the start timestamp of the next version.
    rebuilt_scd2_df = (
        rebuild_source_df
        .withColumn(
            "next_effective_from",
            F.lead("effective_from").over(
                history_window
            )
        )
        .withColumn(
            "effective_to",
            F.col("next_effective_from")
        )
        .withColumn(
            "is_current",
            F.col("next_effective_from").isNull()
        )
        .drop("next_effective_from")
    )

    # Match the required Silver schema.
    new_versions_df = (
        rebuilt_scd2_df
        .select(
            "customer_id",
            "name",
            "city",
            "market",
            "effective_from",
            "effective_to",
            "is_current",
            "ingestion_timestamp",
            "source_file",
            "run_id",
            "batch_id"
        )
    )

    # Validate before creating Silver.
    duplicate_current_df = (
        new_versions_df
        .filter(F.col("is_current") == True)
        .groupBy("customer_id")
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicate_current_df.limit(1).count() > 0:

        display(duplicate_current_df)

        raise Exception(
            "Initial load failed: duplicate current "
            "customer versions found."
        )

    invalid_dates_df = (
        new_versions_df
        .filter(
            F.col("effective_to").isNotNull()
            & (
                F.col("effective_to")
                <= F.col("effective_from")
            )
        )
    )

    if invalid_dates_df.limit(1).count() > 0:

        display(invalid_dates_df.limit(100))

        raise Exception(
            "Initial load failed: invalid SCD2 dates. "
            "Check records with duplicate or conflicting "
            "effective timestamps."
        )

    # Create the Silver Delta table.
    (
        new_versions_df
        .write
        .format("delta")
        .mode("errorifexists")
        .save(silver_path)
    )

    print(
        "Initial Silver history created:",
        new_versions_df.count()
    )


# ============================================================
# 4. INCREMENTAL LOAD - KEEP LATEST INCOMING ROW PER CUSTOMER
# ============================================================

else:

    incoming_window = (
        Window
        .partitionBy("customer_id")
        .orderBy(
            F.col("ingestion_timestamp").desc(),
            F.col("source_file").desc()
        )
    )

    latest_incremental_customers_df = (
        processed_customers_df
        .withColumn(
            "rn",
            F.row_number().over(incoming_window)
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # Read only the current versions from Silver.
    current_customers_df = (
        spark.read
        .format("delta")
        .load(silver_path)
        .filter(F.col("is_current") == True)
        .select(
            "customer_id",
            "name",
            "city",
            "market"
        )
    )

    # Compare incoming records with current Silver records.
    comparison_df = (
        latest_incremental_customers_df.alias("source")
        .join(
            current_customers_df.alias("target"),
            F.col("source.customer_id")
            == F.col("target.customer_id"),
            "left"
        )
        .select(
            F.col("source.*"),
            F.col("target.name").alias("current_name"),
            F.col("target.city").alias("current_city"),
            F.col("target.market").alias("current_market"),
            F.col("target.customer_id").alias(
                "existing_customer_id"
            )
        )
    )

    # --------------------------------------------------------
    # Identify new customers
    # --------------------------------------------------------

    new_customers_df = (
        comparison_df
        .filter(
            F.col("existing_customer_id").isNull()
        )
    )

    # --------------------------------------------------------
    # Identify changed customers
    # --------------------------------------------------------

    changed_customers_df = (
        comparison_df
        .filter(
            F.col("existing_customer_id").isNotNull()
            & (
                ~F.col("name").eqNullSafe(
                    F.col("current_name")
                )
                | ~F.col("city").eqNullSafe(
                    F.col("current_city")
                )
                | ~F.col("market").eqNullSafe(
                    F.col("current_market")
                )
            )
        )
    )

    # --------------------------------------------------------
    # Identify unchanged customers
    # --------------------------------------------------------

    unchanged_customers_df = (
        comparison_df
        .filter(
            F.col("existing_customer_id").isNotNull()
            & F.col("name").eqNullSafe(
                F.col("current_name")
            )
            & F.col("city").eqNullSafe(
                F.col("current_city")
            )
            & F.col("market").eqNullSafe(
                F.col("current_market")
            )
        )
    )

    print("New customers:", new_customers_df.count())
    print("Changed customers:", changed_customers_df.count())
    print("Unchanged customers:", unchanged_customers_df.count())

    # --------------------------------------------------------
    # Expire old versions of changed customers
    # --------------------------------------------------------

    changed_customer_ids_df = (
        changed_customers_df
        .select(
            "customer_id",
            F.col("ingestion_timestamp").alias(
                "new_effective_from"
            )
        )
    )

    if changed_customer_ids_df.limit(1).count() > 0:

        silver_table = DeltaTable.forPath(
            spark,
            silver_path
        )

        (
            silver_table.alias("target")
            .merge(
                changed_customer_ids_df.alias("source"),
                """
                target.customer_id = source.customer_id
                AND target.is_current = true
                """
            )
            .whenMatchedUpdate(
                set={
                    "effective_to": "source.new_effective_from",
                    "is_current": "false"
                }
            )
            .execute()
        )

        print("Previous customer versions expired.")

    # --------------------------------------------------------
    # Prepare new and changed versions for insertion
    # --------------------------------------------------------

    new_versions_df = (
        new_customers_df
        .unionByName(
            changed_customers_df,
            allowMissingColumns=True
        )
        .select(
            "customer_id",
            "name",
            "city",
            "market",
            F.col("ingestion_timestamp").alias(
                "effective_from"
            ),
            F.lit(None).cast("timestamp").alias(
                "effective_to"
            ),
            F.lit(True).alias("is_current"),
            "ingestion_timestamp",
            "source_file",
            "run_id",
            "batch_id"
        )
    )

    new_version_count = new_versions_df.count()

    # Append new customer versions.
    if new_version_count > 0:

        (
            new_versions_df
            .write
            .format("delta")
            .mode("append")
            .save(silver_path)
        )

        print("New SCD2 versions inserted:", new_version_count)

    else:

        print("No new SCD2 versions to insert.")





In [0]:
# ============================================================
# 5. VALIDATE FINAL SILVER TABLE
# ============================================================

final_silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

# Each customer must have at most one current version.
duplicate_current_df = (
    final_silver_df
    .filter(F.col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_current_count = duplicate_current_df.count()

print(
    "Duplicate current customer IDs:",
    duplicate_current_count
)

if duplicate_current_count > 0:

    display(duplicate_current_df)

    raise Exception(
        "SCD2 validation failed: multiple current "
        "versions exist for a customer."
    )

# Historical end dates must be later than start dates.
invalid_dates_df = (
    final_silver_df
    .filter(
        F.col("effective_to").isNotNull()
        & (
            F.col("effective_to")
            <= F.col("effective_from")
        )
    )
)

invalid_dates_count = invalid_dates_df.count()

print("Invalid effective date records:", invalid_dates_count)

if invalid_dates_count > 0:

    display(invalid_dates_df.limit(100))

    raise Exception(
        "SCD2 validation failed: effective_to must be "
        "greater than effective_from."
    )

print("Silver validation passed.")
print("Final Silver record count:", final_silver_df.count())


# ============================================================
# 6. UPDATE WATERMARK AFTER SUCCESSFUL VALIDATION
# ============================================================

processed_timestamp = (
    processed_customers_df
    .agg(
        F.max("ingestion_timestamp").alias(
            "max_ingestion_timestamp"
        )
    )
    .collect()[0]["max_ingestion_timestamp"]
)

if processed_timestamp is None:

    raise Exception(
        "Cannot update watermark: processed timestamp is NULL."
    )

run_id = (
    f"CUSTOMERS_{processed_timestamp.strftime('%Y%m%d%H%M%S')}"
)

final_watermark_df = spark.createDataFrame(
    [
        (
            PIPELINE_NAME,
            processed_timestamp,
            run_id,
            None
        )
    ],
    schema=watermark_schema
).withColumn(
    "updated_at",
    F.current_timestamp()
)

(
    control_table.alias("target")
    .merge(
        final_watermark_df.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Watermark updated:", processed_timestamp)
print("Customer Silver SCD2 completed successfully.")

Duplicate current customer IDs: 0
Invalid effective date records: 0
Silver validation passed.
Final Silver record count: 150003


In [0]:
# ============================================================
# 17. FINAL OUTPUT
# ============================================================

print("================================================")
print("CUSTOMER SILVER PIPELINE COMPLETE")
print("================================================")


final_silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)
print(
    "Silver customer records:",
    final_silver_df.count()
)
print(
    "Current customer records:",
    final_silver_df
    .filter(
        F.col("is_current") == True
    )
    .count()
)
print(
    "Historical customer records:",
    final_silver_df
    .filter(
        F.col("is_current") == False
    )
    .count()
)
print(
    "Final watermark:"
)
display(
    spark.read
    .format("delta")
    .load(control_path)
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
)
print(
    "Sample SCD2 history:"
)
display(
    final_silver_df
    .filter(
        F.col("customer_id") == 100001
    )
    .orderBy("effective_from")
)

CUSTOMER SILVER PIPELINE COMPLETE
Silver customer records: 150003
Current customer records: 145003
Historical customer records: 5000
Final watermark:


pipeline_name,last_processed_timestamp,last_run_id,updated_at
customers_silver,2026-10-14T08:35:16.583Z,"CUSTOMERS_REBUILD_Column<'date_format(current_timestamp(), yyyyMMddHHmmss)'>",2026-10-14T17:32:29.928Z


Sample SCD2 history:


customer_id,name,city,market,effective_from,effective_to,is_current,ingestion_timestamp,source_file,run_id,batch_id
100001,Aarav Sharma,Hyderabad,India,2026-10-14T08:29:49.936Z,null,true,2026-10-14T08:29:49.936Z,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,be2b12b0-8a66-42b7-a98f-3ff2b9861bae,b1bd8ab0-8e65-4009-a8da-2ac857abc3d7
